### Leer todos los datos que son requediros

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
movies_df = spark.read.table("movie_silver.movies") \
    .filter(f"file_date = '{v_file_date}'")

In [0]:
country_df = spark.read.table("movie_silver.countries")

In [0]:
production_country_df = spark.read.table("movie_silver.productions_countries") \
    .filter(f"file_date = '{v_file_date}'")

### Join "country" y "production_country"

In [0]:
country_prod_coun_df = country_df.join(production_country_df,
                                   country_df.country_id == production_country_df.country_id,
                                   "inner") \
                    .select(country_df.country_name,
                            production_country_df.movie_id)

### Join "movies_df" y "country_prod_coun_df"

- Filtrar las películas donde su fecha de lanzamiento sea mayor o igual a 2015

In [0]:
movies_filter_df = movies_df.filter("year_release_date >= 2015")

In [0]:
results_movies_country_prod_count_df = movies_filter_df.join(country_prod_coun_df,
                                                 movies_filter_df.movie_id == country_prod_coun_df.movie_id,
                                                 "inner")

In [0]:
results_df = results_movies_country_prod_count_df.select("year_release_date", "budget", "revenue", "country_name")

In [0]:
from pyspark.sql.functions import sum

results_group_by_df = results_df \
    .groupBy("year_release_date", "country_name") \
    .agg(
        sum("budget").alias("total_budget"),
        sum("revenue").alias("total_revenue")
    )

In [0]:
from pyspark.sql.functions import lit

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import desc, dense_rank

results_dense_rank_df = Window.partitionBy("year_release_date").orderBy(
                                                                desc("total_budget"), 
                                                                desc("total_revenue"))
final_df = results_group_by_df.withColumn("dense_rank", dense_rank().over(results_dense_rank_df)) \
    .withColumn("created_date", lit(v_file_date))

### Escribir datos en el DataLake en formato "Delta"

In [0]:
#overwrite_partition("movie_gold", "results_group_movie_country", "created_date", v_file_date)

In [0]:
merge_condition = 'tgt.year_release_date = src.year_release_date AND tgt.country_name = src.country_name AND tgt.created_date = src.created_date'
merge_delta_lake(final_df, "movie_gold", "results_group_movie_country", merge_condition, "created_date")

In [0]:
%sql
SELECT * FROM movie_gold.results_group_movie_country;

year_release_date,country_name,total_budget,total_revenue,dense_rank,created_date
2015,United States of America,5143775000,1.8450384353E10,1,2024-12-30
2015,United Kingdom,651522360,1.894996027E9,2,2024-12-30
2015,Canada,324500000,1.334394558E9,3,2024-12-30
2015,Germany,309500000,9.5835032E8,4,2024-12-30
2015,China,280000000,8.8867852E8,5,2024-12-30
2015,Australia,213000000,6.79034882E8,6,2024-12-30
2015,Japan,190000000,1.50624936E9,7,2024-12-30
2015,France,151000000,2.06495048E8,8,2024-12-30
2015,Hong Kong,135000000,5.32950503E8,9,2024-12-30
2015,Taiwan,135000000,5.32950503E8,9,2024-12-30
